# Assignment 1 — NumPy: vectorised computation
No Python `for` loop over NumPy array elements is used anywhere in this assignment.


In [5]:
import numpy as np
import pandas as pd
import time
from numpy.lib.stride_tricks import sliding_window_view

np.random.seed(42)

## Q1. Create array A (1000, 5), mean 50, std 12

In [6]:
A = np.random.normal(loc=50, scale=12, size=(1000, 5))
print("Shape :", A.shape)
print("dtype :", A.dtype)
print("Memory (float64):", A.nbytes, "bytes")

A32 = A.astype(np.float32)
print("Memory (float32):", A32.nbytes, "bytes")

Shape : (1000, 5)
dtype : float64
Memory (float64): 40000 bytes
Memory (float32): 20000 bytes


## Q2. Standardise every column (one broadcasting expression)

In [7]:
Z = (A - A.mean(axis=0)) / A.std(axis=0)

print("Column means after scaling:", np.round(Z.mean(axis=0), 6) + 0.0)
print("Column stds  after scaling:", np.round(Z.std(axis=0), 6))

Column means after scaling: [0. 0. 0. 0. 0.]
Column stds  after scaling: [1. 1. 1. 1. 1.]


## Q3. `minmax(X, axis)` scaling to the range 0–1

In [8]:
def minmax(X, axis):
    mn = X.min(axis=axis, keepdims=True)
    mx = X.max(axis=axis, keepdims=True)
    return (X - mn) / (mx - mn)

M0 = minmax(A, axis=0)
M1 = minmax(A, axis=1)

print("axis=0 -> min per column:", M0.min(axis=0))
print("axis=0 -> max per column:", M0.max(axis=0))
print("axis=1 -> min of first 5 rows:", M1.min(axis=1)[:5])
print("axis=1 -> max of first 5 rows:", M1.max(axis=1)[:5])
print("All mins are 0 and all maxes are 1 (axis=0):", np.allclose(M0.min(axis=0), 0) and np.allclose(M0.max(axis=0), 1))
print("All mins are 0 and all maxes are 1 (axis=1):", np.allclose(M1.min(axis=1), 0) and np.allclose(M1.max(axis=1), 1))

axis=0 -> min per column: [0. 0. 0. 0. 0.]
axis=0 -> max per column: [1. 1. 1. 1. 1.]
axis=1 -> min of first 5 rows: [0. 0. 0. 0. 0.]
axis=1 -> max of first 5 rows: [1. 1. 1. 1. 1.]
All mins are 0 and all maxes are 1 (axis=0): True
All mins are 0 and all maxes are 1 (axis=1): True


## Q4. Inject NaN into 5% of positions, then fill with the column mean

In [9]:
n_nan = int(0.05 * A.size)                                   # 250 positions
pos = np.random.choice(A.size, size=n_nan, replace=False)     # random flat positions
A_original = A.copy()                                         # keep a copy of the untouched data
A.flat[pos] = np.nan                                          # inject NaN

print("NaN injected:", np.isnan(A).sum(), "(", 100 * np.isnan(A).mean(), "% )")

col_means = np.nanmean(A, axis=0)                             # mean of each column, ignoring NaN
mask = np.isnan(A)                                            # boolean mask of the NaN positions
A[mask] = np.broadcast_to(col_means, A.shape)[mask]           # each NaN gets its own column's mean

print("Column means used:", np.round(col_means, 4))
print("NaN remaining:", np.isnan(A).sum())
assert np.isnan(A).sum() == 0

NaN injected: 250 ( 5.0 % )
Column means used: [49.8838 49.7273 50.2018 50.0886 50.4297]
NaN remaining: 0


## Q5. Replace values beyond 3 standard deviations with the column median

In [10]:
mu  = A.mean(axis=0)
sd  = A.std(axis=0)
med = np.median(A, axis=0)

outlier_mask = np.abs(A - mu) > 3 * sd
print("Values replaced per column:", outlier_mask.sum(axis=0))
print("Total replaced:", outlier_mask.sum())

A = np.where(outlier_mask, med, A)

Values replaced per column: [5 1 5 3 5]
Total replaced: 19


## Q6. 200 × 200 Euclidean distance matrix (broadcasting only)

In [11]:
X = A[:200]                                   # (200, 5)
diff = X[:, None, :] - X[None, :, :]          # (200, 1, 5) - (1, 200, 5) -> (200, 200, 5)
D = np.sqrt((diff ** 2).sum(axis=-1))         # (200, 200)

print("Shape:", D.shape)
print("Diagonal is all 0 :", np.allclose(np.diag(D), 0))
print("D equals its transpose:", np.array_equal(D, D.T))

Shape: (200, 200)
Diagonal is all 0 : True
D equals its transpose: True


## Q7. 5 nearest rows per row with `argpartition`

In [12]:
D2 = D.copy()
np.fill_diagonal(D2, np.inf)                  # a row must not count itself as its own neighbour

nn_idx = np.argpartition(D2, kth=4, axis=1)[:, :5]

print("Shape:", nn_idx.shape, "| dtype:", nn_idx.dtype)
print("First 3 rows:\n", nn_idx[:3])

# sanity check against a full sort (same neighbours, ordering ignored)
check = np.argsort(D2, axis=1)[:, :5]
print("Matches full-sort result:", np.array_equal(np.sort(nn_idx, axis=1), np.sort(check, axis=1)))
print("No row lists itself:", not (nn_idx == np.arange(200)[:, None]).any())

Shape: (200, 5) | dtype: int64
First 3 rows:
 [[180 179 116 178 192]
 [114  17  31 112 197]
 [  3 158 132 145 102]]
Matches full-sort result: True
No row lists itself: True


**Why `argpartition`:** it only separates the 5 smallest distances from the rest in about O(n) time per row, while a full sort does O(n log n) work to order values we never need ordered.

## Q8. `np.bincount` vs a plain Python loop (1,000,000 integers)
**Note:** the assignment asks for a plain Python loop here, but also bans loops over NumPy elements. To respect both, the loop runs over a normal Python *list* (`arr.tolist()`), not over the NumPy array. The conversion is done in the setup cell, outside the timed cells.

In [20]:
arr = np.random.randint(0, 100, size=1_000_000)
arr_list = arr.tolist()          # plain Python list used only by the loop version
print(arr.shape, arr.min(), arr.max())

(1000000,) 0 99


In [21]:
%%time
t0 = time.perf_counter()
counts_np = np.bincount(arr, minlength=100)
t_np = time.perf_counter() - t0

CPU times: user 3.43 ms, sys: 0 ns, total: 3.43 ms
Wall time: 2.6 ms


In [15]:
%%time
t0 = time.perf_counter()
counts_loop = [0] * 100
for v in arr_list:
    counts_loop[v] += 1
t_loop = time.perf_counter() - t0

CPU times: user 136 ms, sys: 0 ns, total: 136 ms
Wall time: 139 ms


In [16]:
print("Both methods give the same counts:", np.array_equal(counts_np, np.array(counts_loop)))
print(f"bincount time : {t_np:.6f} s")
print(f"loop time     : {t_loop:.6f} s")
speedup = t_loop / t_np
print(f"SPEED-UP      : {speedup:.1f}x")

Both methods give the same counts: True
bincount time : 0.002421 s
loop time     : 0.138769 s
SPEED-UP      : 57.3x


## Q9. Sliding windows of length 10 over 10,000 values

In [17]:
x = np.random.normal(size=10_000)

W = sliding_window_view(x, window_shape=10)
print("Windows shape:", W.shape)

win_mean = W.mean(axis=1)
pd_mean = pd.Series(x).rolling(10).mean().to_numpy()   # first 9 entries are NaN

print("Matches pandas rolling mean:", np.allclose(win_mean, pd_mean[9:]))

Windows shape: (9991, 10)
Matches pandas rolling mean: True


## Q10. Reshape (12, 8) → (4, 3, 8) and average over the middle axis

In [18]:
B = np.arange(96).reshape(12, 8)
B3 = B.reshape(4, 3, 8)
R = B3.mean(axis=1)

print("Reshaped:", B3.shape, "-> after mean over axis 1:", R.shape)
print(R)

Reshaped: (4, 3, 8) -> after mean over axis 1: (4, 8)
[[ 8.  9. 10. 11. 12. 13. 14. 15.]
 [32. 33. 34. 35. 36. 37. 38. 39.]
 [56. 57. 58. 59. 60. 61. 62. 63.]
 [80. 81. 82. 83. 84. 85. 86. 87.]]


The 12 rows are grouped into 4 blocks of 3 consecutive rows, and each block is averaged into one row.
So the result `(4, 8)` holds 4 block-means, each still with all 8 columns.

## Q11. Save and reload the cleaned array

In [19]:
np.save("clean.npy", A)
A_loaded = np.load("clean.npy")

assert np.array_equal(A, A_loaded)
print("Loaded array is identical to the original:", np.array_equal(A, A_loaded))

Loaded array is identical to the original: True
